# ML Lab 5
Naive Bayes and Decision Tree on Breast Cancer dataset.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, precision_recall_curve, roc_curve, auc

data=load_breast_cancer(); X=pd.DataFrame(data.data,columns=data.feature_names); y=pd.Series(data.target)
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

# Experiment 1: Naive Bayes
nb=GaussianNB().fit(X_train,y_train)
y_nb=nb.predict(X_test); p_nb=nb.predict_proba(X_test)[:,1]
acc_nb_tr=accuracy_score(y_train,nb.predict(X_train)); acc_nb_te=accuracy_score(y_test,y_nb)
cm_nb=confusion_matrix(y_test,y_nb)
print('Naive Bayes train/test accuracy:',round(acc_nb_tr,4),round(acc_nb_te,4))
print(classification_report(y_test,y_nb,target_names=data.target_names))

# Experiment 2: Decision Tree
dt=DecisionTreeClassifier(random_state=42,max_depth=4).fit(X_train,y_train)
y_dt=dt.predict(X_test); p_dt=dt.predict_proba(X_test)[:,1]
acc_dt_tr=accuracy_score(y_train,dt.predict(X_train)); acc_dt_te=accuracy_score(y_test,y_dt)
cm_dt=confusion_matrix(y_test,y_dt)
print('Decision Tree train/test accuracy:',round(acc_dt_tr,4),round(acc_dt_te,4))
print(classification_report(y_test,y_dt,target_names=data.target_names))

# PR curves
pr_nb, rc_nb, _ = precision_recall_curve(y_test,p_nb)
pr_dt, rc_dt, _ = precision_recall_curve(y_test,p_dt)
plt.figure(figsize=(6,4)); plt.plot(rc_nb,pr_nb,label='Naive Bayes'); plt.plot(rc_dt,pr_dt,label='Decision Tree'); plt.xlabel('Recall'); plt.ylabel('Precision'); plt.title('Precision-Recall Curves'); plt.legend(); plt.show()

# Decision tree visualization
plt.figure(figsize=(20,8)); plot_tree(dt,feature_names=X.columns,class_names=data.target_names,filled=True,fontsize=7); plt.show()

# Experiment 3: comparisons
fpr_nb,tpr_nb,_=roc_curve(y_test,p_nb); fpr_dt,tpr_dt,_=roc_curve(y_test,p_dt)
auc_nb=auc(fpr_nb,tpr_nb); auc_dt=auc(fpr_dt,tpr_dt)

plt.figure(figsize=(6,4));
x=np.arange(2); w=0.35
plt.bar(x-w/2,[acc_nb_tr,acc_dt_tr],w,label='Train'); plt.bar(x+w/2,[acc_nb_te,acc_dt_te],w,label='Test');
plt.xticks(x,['Naive Bayes','Decision Tree']); plt.ylim(0,1.05); plt.title('Accuracy Comparison'); plt.legend(); plt.show()

plt.figure(figsize=(6,4)); plt.plot(fpr_nb,tpr_nb,label=f'NB AUC={auc_nb:.3f}'); plt.plot(fpr_dt,tpr_dt,label=f'DT AUC={auc_dt:.3f}'); plt.plot([0,1],[0,1],'--',color='gray'); plt.xlabel('FPR'); plt.ylabel('TPR'); plt.title('ROC Comparison'); plt.legend(); plt.show()

fig,ax=plt.subplots(1,2,figsize=(10,4)); sns.heatmap(cm_nb,annot=True,fmt='d',cmap='Blues',ax=ax[0]); ax[0].set_title('NB Confusion Matrix'); sns.heatmap(cm_dt,annot=True,fmt='d',cmap='Greens',ax=ax[1]); ax[1].set_title('DT Confusion Matrix'); plt.tight_layout(); plt.show()

print(pd.DataFrame({'Model':['Naive Bayes','Decision Tree'],'Train Accuracy':[acc_nb_tr,acc_dt_tr],'Test Accuracy':[acc_nb_te,acc_dt_te],'ROC-AUC':[auc_nb,auc_dt]}))


## Answers
1. Higher accuracy: Decision Tree (typical run).
2. Better recall for malignant cases: Naive Bayes (typical run).
3. More overfitting-prone: Decision Tree, because it can memorize training splits.
4. Most important metric in diagnosis: Recall, to reduce false negatives.